# XGBoost NFL Running Back Performance Prediction

This notebook demonstrates how LLM-engineered features can enhance traditional ML models.

**Goal**: Predict weekly RB performance using:

- Statistical features (yards, touches, opponent rank)
- LLM-generated features (press ratings, injury likelihood, intuition grades)


In [130]:
%pip install nflreadpy anthropic python-dotenv pyarrow matplotlib xgboost scikit-learn

import nflreadpy as nflread
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import anthropic
import os
import pyarrow as pa

from dotenv import load_dotenv

load_dotenv()

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


True

## 1. Load NFL Running Back Data


In [131]:
# Load weekly player stats for 2025 season using nflreadpy
print("Loading 2025 NFL season data...")
weekly_stats_polars = nflread.load_player_stats([2025])

print(f"Data type: {type(weekly_stats_polars)}")
print(f"Shape: {weekly_stats_polars.shape}")

# Convert Polars to Pandas without pyarrow - use dict method
# This avoids the pyarrow dependency issue
weekly_stats = pd.DataFrame(weekly_stats_polars.to_dict(as_series=False))

# Filter for running backs only
rb_stats = weekly_stats[weekly_stats["position"] == "RB"].copy()

# First, let's check what columns are available
print("\nAvailable columns:")
print([col for col in rb_stats.columns if "team" in col.lower()])

# Select relevant statistical columns - using 'team' instead of 'recent_team'
stat_columns = [
    "player_id",
    "player_name",
    "week",
    "season",
    "rushing_yards",
    "rushing_tds",
    "carries",
    "targets",
    "receptions",
    "receiving_yards",
    "receiving_tds",
    "fantasy_points_ppr",
    "opponent_team",
    "team",
]

rb_stats = rb_stats[stat_columns].copy()

# Remove rows with missing rushing yards (our target)
rb_stats = rb_stats.dropna(subset=["rushing_yards"])

print(f"\nLoaded {len(rb_stats)} RB performances from 2025 season")
print(f"Unique players: {rb_stats['player_name'].nunique()}")
print(f"Weeks covered: {rb_stats['week'].min()} to {rb_stats['week'].max()}")

rb_stats.head()

Loading 2025 NFL season data...
Data type: <class 'polars.dataframe.frame.DataFrame'>
Shape: (4462, 114)

Available columns:
['team', 'opponent_team', 'special_teams_tds']

Loaded 375 RB performances from 2025 season
Unique players: 113
Weeks covered: 1 to 5


,player_id,player_name,week,season,rushing_yards,rushing_tds,carries,targets,receptions,receiving_yards,receiving_tds,fantasy_points_ppr,opponent_team,team
77,00-0032764,D.Henry,1,2025,169,2,18,1,1,13,0,29.2,BUF,BAL
96,00-0033280,C.McCaffrey,1,2025,69,0,22,10,9,73,0,23.2,SEA,SF
100,00-0033293,A.Jones,1,2025,23,0,8,3,3,44,1,15.7,CHI,MIN
108,00-0033526,S.Perine,1,2025,0,0,0,2,2,6,0,2.6,CLE,CIN
112,00-0033553,J.Conner,1,2025,39,0,12,4,4,5,1,14.4,NO,ARI


## 2. Engineer Basic Statistical Features


In [132]:
# Sort by player and week
rb_stats = rb_stats.sort_values(["player_id", "week"]).reset_index(drop=True)

# Create lagged features (previous week performance)
rb_stats["prev_rushing_yards"] = rb_stats.groupby("player_id")["rushing_yards"].shift(1)
rb_stats["prev_carries"] = rb_stats.groupby("player_id")["carries"].shift(1)
rb_stats["prev_fantasy_points"] = rb_stats.groupby("player_id")[
    "fantasy_points_ppr"
].shift(1)

# Rolling averages (last 3 weeks)
rb_stats["avg_rushing_yards_3w"] = rb_stats.groupby("player_id")[
    "rushing_yards"
].transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
rb_stats["avg_carries_3w"] = rb_stats.groupby("player_id")["carries"].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

# Target variable: rushing yards this week
rb_stats["target_rushing_yards"] = rb_stats["rushing_yards"]

# Filter for weeks 2-4 only
# - Week 1 dropped due to missing lagged features (no "previous week" data)
# - Week 5 excluded as games haven't occurred yet (Oct 4, 2025)
rb_stats = rb_stats[rb_stats["week"].isin([2, 3, 4])].copy()

print(f"After feature engineering: {len(rb_stats)} rows (weeks 2-4 only)")
print(f"Week 1: Dropped (no lagged features)")
print(f"Week 5: Excluded (games haven't happened yet)")
rb_stats.head()

After feature engineering: 274 rows (weeks 2-4 only)
Week 1: Dropped (no lagged features)
Week 5: Excluded (games haven't happened yet)


,player_id,player_name,week,season,rushing_yards,rushing_tds,carries,targets,receptions,receiving_yards,receiving_tds,fantasy_points_ppr,opponent_team,team,prev_rushing_yards,prev_carries,prev_fantasy_points,avg_rushing_yards_3w,avg_carries_3w,target_rushing_yards
0,00-0031687,R.Mostert,4,2025,62,0,4,1,1,11,0,8.3,CHI,LV,NaN,NaN,NaN,NaN,NaN,62
2,00-0032764,D.Henry,2,2025,23,0,11,0,0,0,0,2.3,CLE,BAL,169.0,18.0,29.2,169.000000,18.000000,23
3,00-0032764,D.Henry,3,2025,50,1,12,1,1,7,0,10.7,DET,BAL,23.0,11.0,2.3,96.000000,14.500000,50
4,00-0032764,D.Henry,4,2025,42,0,8,3,2,16,0,7.8,KC,BAL,50.0,12.0,10.7,80.666667,13.666667,42
6,00-0033280,C.McCaffrey,2,2025,55,0,13,7,6,52,1,22.7,NO,SF,69.0,22.0,23.2,69.000000,22.000000,55


## 3. LLM Feature Engineering

Now we'll use Claude to generate abstract features that capture qualitative information:

- **press_rating**: 1-10 rating based on recent news sentiment
- **injury_concern**: 1-5 scale of injury risk
- **intuition_grade**: 1-5 LLM assessment based on historical pattern recognition
- **opponent_defense_rating**: 1-10 rating of opponent run defense (higher = weaker defense)
- **oline_health**: 1-5 rating of offensive line health
- **vegas_sentiment**: 1-10 rating based on betting lines and expert picks


In [133]:
# Initialize Anthropic client
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

# We'll use a smaller sample for LLM feature generation (expensive operation)
# Focus on players with significant playing time
significant_rbs = (
    rb_stats.groupby("player_id")
    .agg({"carries": "sum", "player_name": "first"})
    .reset_index()
)

significant_rbs = significant_rbs[significant_rbs["carries"] >= 10].sort_values(
    "carries", ascending=False
)
print(f"Focusing on {len(significant_rbs)} RBs with 10+ carries in 2025")
print(significant_rbs.head(10))

Focusing on 60 RBs with 10+ carries in 2025
     player_id  carries player_name
44  00-0037248       62      J.Cook
21  00-0035700       61    J.Jacobs
25  00-0036223       59    J.Taylor
12  00-0034844       59   S.Barkley
75  00-0039361       57    B.Irving
8   00-0033906       54    A.Kamara
58  00-0038542       52  B.Robinson
53  00-0037840       50  K.Williams
17  00-0035261       50   T.Pollard
90  00-0040122       49    A.Jeanty


In [134]:
import json
import os
from pathlib import Path
import re
import asyncio
from anthropic import AsyncAnthropic

# Create cache directory
CACHE_DIR = Path("llm_feature_cache")
CACHE_DIR.mkdir(exist_ok=True)

# Cache version - increment when prompts change
CACHE_VERSION = "v1"

# Initialize async client
async_client = AsyncAnthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))


def get_cache_key(feature_type, **kwargs):
    """Generate a unique cache key for a feature request"""
    # Sanitize values for filesystem safety
    sanitized_kwargs = {}
    for k, v in kwargs.items():
        # Remove special characters, keep only alphanumeric, spaces, hyphens
        sanitized_kwargs[k] = re.sub(r"[^\w\s-]", "", str(v))

    key_parts = [CACHE_VERSION, feature_type] + [
        f"{k}={v}" for k, v in sorted(sanitized_kwargs.items())
    ]
    return "_".join(str(p).replace(" ", "_") for p in key_parts) + ".json"


def load_from_cache(feature_type, **kwargs):
    """Load a cached feature value if it exists"""
    cache_file = CACHE_DIR / get_cache_key(feature_type, **kwargs)
    if cache_file.exists():
        with open(cache_file, "r") as f:
            data = json.load(f)
            print(f"  [CACHE HIT] Loaded {feature_type} from cache")
            return data["value"]
    return None


def save_to_cache(feature_type, value, is_default=False, **kwargs):
    """Save a feature value to cache"""
    cache_file = CACHE_DIR / get_cache_key(feature_type, **kwargs)
    with open(cache_file, "w") as f:
        json.dump(
            {"value": value, "kwargs": kwargs, "is_default": is_default}, f, indent=2
        )


async def get_llm_press_rating_async(player_name, week, year=2025):
    """Use Claude with web search to rate player press sentiment"""
    # Check cache first
    cached = load_from_cache(
        "press_rating", player_name=player_name, week=week, year=year
    )
    if cached is not None:
        return cached

    prompt = f"""
    Search for recent news and sentiment about NFL running back {player_name}
    around week {week} of the {year} season.

    Based on the press coverage, rate the player's public perception on a scale of 1-10:
    - 1-3: Negative coverage (injury concerns, poor performance, controversy)
    - 4-6: Neutral or mixed coverage
    - 7-10: Positive coverage (breakout performance, healthy, favorable matchup)

    First, briefly explain what you found in the search results (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 10.
    """

    try:
        response = await async_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
            tools=[
                {"type": "web_search_20250305", "name": "web_search", "max_uses": 3}
            ],
        )

        # Extract text from response
        text_content = ""
        for block in response.content:
            if block.type == "text":
                text_content += block.text

        # Try to extract the number from the last line
        lines = text_content.strip().split("\n")
        rating = None
        for line in reversed(lines):
            try:
                rating = int(line.strip())
                if 1 <= rating <= 10:
                    break
            except:
                continue
        result = rating if rating else 5
    except Exception as e:
        print(f"  Error in press_rating: {e}")
        result = 5  # Default neutral rating

    # Save to cache
    save_to_cache(
        "press_rating",
        result,
        is_default=(result == 5),
        player_name=player_name,
        week=week,
        year=year,
    )
    return result


async def get_llm_injury_concern_async(player_name, week, year=2025):
    """Use Claude with web search to assess injury likelihood"""
    # Check cache first
    cached = load_from_cache(
        "injury_concern", player_name=player_name, week=week, year=year
    )
    if cached is not None:
        return cached

    prompt = f"""
    Search for injury reports about NFL running back {player_name}
    around week {week} of the {year} season.

    Rate the injury concern level on a scale of 1-5:
    - 1: No injury concerns, fully healthy
    - 2: Minor issue, questionable but likely to play
    - 3: Moderate concern, may be limited
    - 4: Significant concern, doubtful to play
    - 5: Out or ruled out

    First, briefly explain what you found in injury reports (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 5.
    """

    try:
        response = await async_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
            tools=[
                {"type": "web_search_20250305", "name": "web_search", "max_uses": 3}
            ],
        )

        # Extract text from response
        text_content = ""
        for block in response.content:
            if block.type == "text":
                text_content += block.text

        lines = text_content.strip().split("\n")
        rating = None
        for line in reversed(lines):
            try:
                rating = int(line.strip())
                if 1 <= rating <= 5:
                    break
            except:
                continue
        result = rating if rating else 1
    except Exception as e:
        print(f"  Error in injury_concern: {e}")
        result = 1  # Default healthy

    # Save to cache
    save_to_cache(
        "injury_concern",
        result,
        is_default=(result == 1),
        player_name=player_name,
        week=week,
        year=year,
    )
    return result


async def get_llm_intuition_grade_async(player_data_json):
    """Use Claude to provide an intuition-based grade on player's trajectory"""
    # Check cache first - use hash of player_data_json as key
    import hashlib

    data_hash = hashlib.md5(player_data_json.encode()).hexdigest()
    cached = load_from_cache("intuition_grade", data_hash=data_hash)
    if cached is not None:
        return cached

    prompt = f"""
    You are an expert NFL analyst. Review this running back's recent performance data:

    {player_data_json}

    Based on patterns, trends, and your expertise, give an intuition grade for their
    NEXT game performance on a scale of 1-5:
    - 1: Expect poor performance
    - 2: Below average expected
    - 3: Average expected
    - 4: Above average expected
    - 5: Breakout performance expected

    Consider workload trends, efficiency, recent game script, and momentum.
    Respond with ONLY a single number between 1 and 5.
    """

    try:
        response = await async_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=50,
            messages=[{"role": "user", "content": prompt}],
        )

        # Extract text from response
        text_content = ""
        for block in response.content:
            if block.type == "text":
                text_content += block.text

        rating = int(text_content.strip())
        result = max(1, min(5, rating))
    except Exception as e:
        print(f"  Error in intuition_grade: {e}")
        result = 3  # Default average

    # Save to cache
    save_to_cache(
        "intuition_grade", result, is_default=(result == 3), data_hash=data_hash
    )
    return result


async def get_llm_opponent_defense_rating_async(opponent_team, week, year=2025):
    """Use Claude with web search to rate opponent run defense strength"""
    # Check cache first
    cached = load_from_cache(
        "opponent_defense", opponent_team=opponent_team, week=week, year=year
    )
    if cached is not None:
        return cached

    prompt = f"""
    Search for information about the {opponent_team} run defense 
    around week {week} of the {year} NFL season.

    Rate their run defense strength on a scale of 1-10:
    - 1-3: Elite run defense (top ranked, healthy, tough matchup for RBs)
    - 4-6: Average run defense
    - 7-10: Weak run defense (injuries, poor ranking, favorable for RBs)

    First, briefly explain what you found about their run defense (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 10.
    """

    try:
        response = await async_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
            tools=[
                {"type": "web_search_20250305", "name": "web_search", "max_uses": 3}
            ],
        )

        text_content = ""
        for block in response.content:
            if block.type == "text":
                text_content += block.text

        lines = text_content.strip().split("\n")
        rating = None
        for line in reversed(lines):
            try:
                rating = int(line.strip())
                if 1 <= rating <= 10:
                    break
            except:
                continue
        result = rating if rating else 5
    except Exception as e:
        print(f"  Error in opponent_defense: {e}")
        result = 5  # Default average

    # Save to cache
    save_to_cache(
        "opponent_defense",
        result,
        is_default=(result == 5),
        opponent_team=opponent_team,
        week=week,
        year=year,
    )
    return result


async def get_llm_oline_health_async(team, week, year=2025):
    """Use Claude with web search to assess offensive line health"""
    # Check cache first
    cached = load_from_cache("oline_health", team=team, week=week, year=year)
    if cached is not None:
        return cached

    prompt = f"""
    Search for offensive line injury reports for the {team} 
    around week {week} of the {year} NFL season.

    Rate the offensive line health on a scale of 1-5:
    - 1: Multiple starters out or questionable, severe injuries
    - 2: One starter out, or multiple backups playing
    - 3: Minor injuries, some game-time decisions
    - 4: Mostly healthy, minor issues only
    - 5: Fully healthy, all starters playing

    First, briefly explain what you found in injury reports (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 5.
    """

    try:
        response = await async_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
            tools=[
                {"type": "web_search_20250305", "name": "web_search", "max_uses": 3}
            ],
        )

        text_content = ""
        for block in response.content:
            if block.type == "text":
                text_content += block.text

        lines = text_content.strip().split("\n")
        rating = None
        for line in reversed(lines):
            try:
                rating = int(line.strip())
                if 1 <= rating <= 5:
                    break
            except:
                continue
        result = rating if rating else 4
    except Exception as e:
        print(f"  Error in oline_health: {e}")
        result = 4  # Default mostly healthy

    # Save to cache
    save_to_cache(
        "oline_health",
        result,
        is_default=(result == 4),
        team=team,
        week=week,
        year=year,
    )
    return result


async def get_llm_vegas_sentiment_async(player_name, week, year=2025):
    """Use Claude with web search to assess Vegas and expert sentiment"""
    # Check cache first
    cached = load_from_cache(
        "vegas_sentiment", player_name=player_name, week=week, year=year
    )
    if cached is not None:
        return cached

    prompt = f"""
    Search for betting lines, prop lines, and expert picks for NFL running back {player_name}
    for week {week} of the {year} season.

    Rate the Vegas/expert sentiment on a scale of 1-10:
    - 1-3: Bearish - low prop lines, experts fading, unfavorable odds
    - 4-6: Neutral - average expectations
    - 7-10: Bullish - high prop lines, experts hyping, favorable odds

    First, briefly explain what you found about betting lines and expert picks (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 10.
    """

    try:
        response = await async_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
            tools=[
                {"type": "web_search_20250305", "name": "web_search", "max_uses": 3}
            ],
        )

        text_content = ""
        for block in response.content:
            if block.type == "text":
                text_content += block.text

        lines = text_content.strip().split("\n")
        rating = None
        for line in reversed(lines):
            try:
                rating = int(line.strip())
                if 1 <= rating <= 10:
                    break
            except:
                continue
        result = rating if rating else 5
    except Exception as e:
        print(f"  Error in vegas_sentiment: {e}")
        result = 5  # Default neutral

    # Save to cache
    save_to_cache(
        "vegas_sentiment",
        result,
        is_default=(result == 5),
        player_name=player_name,
        week=week,
        year=year,
    )
    return result


async def get_all_llm_features_for_row(row):
    """Get all 6 LLM features for a single row in parallel"""
    player_name = row["player_name"]
    week = row["week"]
    year = row["season"]
    opponent = row["opponent_team"]
    team = row["team"]

    # Get historical data for intuition grade
    player_history = rb_stats[
        (rb_stats["player_id"] == row["player_id"]) & (rb_stats["week"] < week)
    ][["week", "rushing_yards", "carries", "fantasy_points_ppr"]].tail(4)

    player_data_json = player_history.to_json(orient="records")

    # Run all 6 feature calls in parallel
    results = await asyncio.gather(
        get_llm_press_rating_async(player_name, week, year),
        get_llm_injury_concern_async(player_name, week, year),
        get_llm_intuition_grade_async(player_data_json),
        get_llm_opponent_defense_rating_async(opponent, week, year),
        get_llm_oline_health_async(team, week, year),
        get_llm_vegas_sentiment_async(player_name, week, year),
        return_exceptions=True,  # Don't fail entire batch if one fails
    )

    # Handle any exceptions
    default_values = [5, 1, 3, 5, 4, 5]
    processed_results = []
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            print(f"  Error in feature {i}: {result}")
            processed_results.append(default_values[i])
        else:
            processed_results.append(result)

    return processed_results

In [135]:
# Generate LLM features for our significant RBs
# Using async/parallel processing with max 5 concurrent API calls

# Initialize columns
rb_stats["press_rating"] = np.nan
rb_stats["injury_concern"] = np.nan
rb_stats["intuition_grade"] = np.nan
rb_stats["opponent_defense_rating"] = np.nan
rb_stats["oline_health"] = np.nan
rb_stats["vegas_sentiment"] = np.nan

# Filter for weeks 2-4 only (week 5 hasn't happened yet as of Oct 4, 2025)
# Also filter for significant RBs (10+ carries)
top_players = significant_rbs["player_id"].tolist()

rows_to_process = rb_stats[
    (rb_stats["player_id"].isin(top_players))
    & (rb_stats["week"].isin([2, 3, 4]))  # Only weeks 2-4
].copy()

print(f"Generating LLM features for {len(rows_to_process)} player-week samples")
print(f"Weeks: 2-4 (week 5 excluded as games haven't occurred yet)")
print(f"Players: {len(top_players)} RBs with 10+ carries")
print(f"Processing with max 5 concurrent API calls to respect rate limits\\n")


async def process_all_rows_with_limit(rows_df, max_concurrent=5):
    """Process all rows with a limit on concurrent API calls"""
    semaphore = asyncio.Semaphore(max_concurrent)

    async def process_with_semaphore(idx, row):
        async with semaphore:
            player_name = row["player_name"]
            week = row["week"]
            print(f"Processing {player_name} - Week {week}")
            results = await get_all_llm_features_for_row(row)
            return idx, results

    # Create tasks for all rows
    tasks = [process_with_semaphore(idx, row) for idx, row in rows_df.iterrows()]

    # Run all tasks with semaphore limiting concurrency
    results = await asyncio.gather(*tasks)
    return results


# Run the async processing
import nest_asyncio

nest_asyncio.apply()  # Allow nested event loops in Jupyter

results = await process_all_rows_with_limit(rows_to_process, max_concurrent=5)

# Update the dataframe with results
feature_names = [
    "press_rating",
    "injury_concern",
    "intuition_grade",
    "opponent_defense_rating",
    "oline_health",
    "vegas_sentiment",
]

for idx, feature_values in results:
    for feature_name, value in zip(feature_names, feature_values):
        rb_stats.at[idx, feature_name] = value

print(f"\\nLLM feature generation complete!")
print(f"Processed {len(results)} player-week samples")
print(f"Features generated: {feature_names}")

Generating LLM features for 173 player-week samples
Weeks: 2-4 (week 5 excluded as games haven't occurred yet)
Players: 60 RBs with 10+ carries
Processing with max 5 concurrent API calls to respect rate limits\n
Processing D.Henry - Week 2
Processing D.Henry - Week 3
Processing D.Henry - Week 4
Processing C.McCaffrey - Week 2
Processing C.McCaffrey - Week 3
  [CACHE HIT] Loaded press_rating from cache
  [CACHE HIT] Loaded injury_concern from cache
  [CACHE HIT] Loaded intuition_grade from cache
  [CACHE HIT] Loaded opponent_defense from cache
  [CACHE HIT] Loaded oline_health from cache
  [CACHE HIT] Loaded vegas_sentiment from cache
  [CACHE HIT] Loaded press_rating from cache
  [CACHE HIT] Loaded injury_concern from cache
  [CACHE HIT] Loaded intuition_grade from cache
  [CACHE HIT] Loaded opponent_defense from cache
  [CACHE HIT] Loaded oline_health from cache
  [CACHE HIT] Loaded vegas_sentiment from cache
  [CACHE HIT] Loaded press_rating from cache
  [CACHE HIT] Loaded injury_con

In [136]:
# Check the data with new features
rb_stats_with_llm = rb_stats[rb_stats["press_rating"].notna()].copy()
print(f"Samples with LLM features: {len(rb_stats_with_llm)}")
rb_stats_with_llm[
    [
        "player_name",
        "week",
        "rushing_yards",
        "press_rating",
        "injury_concern",
        "intuition_grade",
        "opponent_defense_rating",
        "oline_health",
        "vegas_sentiment",
    ]
].head(10)

Samples with LLM features: 173


,player_name,week,rushing_yards,press_rating,injury_concern,intuition_grade,opponent_defense_rating,oline_health,vegas_sentiment
2,D.Henry,2,23,5.0,1.0,3.0,2.0,4.0,5.0
3,D.Henry,3,50,5.0,1.0,2.0,2.0,4.0,7.0
4,D.Henry,4,42,5.0,1.0,3.0,5.0,4.0,5.0
6,C.McCaffrey,2,55,5.0,1.0,3.0,8.0,3.0,5.0
7,C.McCaffrey,3,52,5.0,1.0,3.0,2.0,4.0,5.0
8,C.McCaffrey,4,49,5.0,1.0,3.0,5.0,3.0,5.0
17,J.Conner,2,34,5.0,1.0,3.0,8.0,4.0,5.0
18,J.Conner,3,22,2.0,1.0,2.0,5.0,3.0,4.0
25,A.Kamara,2,99,3.0,1.0,3.0,3.0,3.0,6.0
26,A.Kamara,3,42,5.0,1.0,3.0,5.0,4.0,5.0


## 4. Train XGBoost Model


In [137]:
# Prepare dataset with LLM features
model_data = rb_stats_with_llm.copy()

# Statistical features
stat_features = [
    "prev_rushing_yards",
    "prev_carries",
    "prev_fantasy_points",
    "avg_rushing_yards_3w",
    "avg_carries_3w",
]

# LLM features
llm_features = [
    "press_rating",
    "injury_concern",
    "intuition_grade",
    "opponent_defense_rating",
    "oline_health",
    "vegas_sentiment",
]

all_features = stat_features + llm_features
target = "target_rushing_yards"

# Remove rows with missing values
model_data = model_data[all_features + [target]].dropna()

print(f"Training samples: {len(model_data)}")
print(f"Statistical features: {stat_features}")
print(f"LLM features: {llm_features}")
print(f"Target: {target}")

Training samples: 170
Statistical features: ['prev_rushing_yards', 'prev_carries', 'prev_fantasy_points', 'avg_rushing_yards_3w', 'avg_carries_3w']
LLM features: ['press_rating', 'injury_concern', 'intuition_grade', 'opponent_defense_rating', 'oline_health', 'vegas_sentiment']
Target: target_rushing_yards


In [138]:
# Train/test split (chronological - use earlier weeks for training)
# For 2025 season with limited weeks, we'll use weeks 2-3 for training and week 4 for testing
model_data_with_week = rb_stats_with_llm.copy()

# Check what weeks we have
print(f"Available weeks: {sorted(model_data_with_week['week'].unique())}")
print(f"Week counts:\n{model_data_with_week['week'].value_counts().sort_index()}")

# Use explicit week split: train on weeks 2-3, test on week 4
# This is more robust for early season data
train_weeks = [2, 3]
test_weeks = [4]

train_indices = model_data_with_week[
    model_data_with_week["week"].isin(train_weeks)
].index
test_indices = model_data_with_week[model_data_with_week["week"].isin(test_weeks)].index

# Now filter model_data to only include features + target
model_data = model_data_with_week[all_features + [target]].dropna()

# Split based on indices that exist in model_data
train_data = model_data.loc[model_data.index.intersection(train_indices)]
test_data = model_data.loc[model_data.index.intersection(test_indices)]

X_train = train_data[all_features]
y_train = train_data[target]
X_test = test_data[all_features]
y_test = test_data[target]

print(f"\nTrain set (weeks {train_weeks}): {len(X_train)} samples")
print(f"Test set (weeks {test_weeks}): {len(X_test)} samples")

# Warn if test set is too small
if len(X_test) < 20:
    print(
        f"\n⚠️  WARNING: Test set is small ({len(X_test)} samples). Results may be less reliable."
    )
if len(X_train) < 20:
    print(
        f"\n⚠️  WARNING: Train set is small ({len(X_train)} samples). Model may underfit."
    )

Available weeks: [2, 3, 4]
Week counts:
2    58
3    59
4    56
Name: week, dtype: int64

Train set (weeks [2, 3]): 114 samples
Test set (weeks [4]): 56 samples


In [139]:
# Train XGBoost model
model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
)

model.fit(X_train, y_train)

# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Evaluate
print("\n=== Model Performance ===")
print(f"\nTrain Set:")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.2f}")
print(f"  MAE: {mean_absolute_error(y_train, y_pred_train):.2f}")
print(f"  R²: {r2_score(y_train, y_pred_train):.3f}")

print(f"\nTest Set:")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}")
print(f"  MAE: {mean_absolute_error(y_test, y_pred_test):.2f}")
print(f"  R²: {r2_score(y_test, y_pred_test):.3f}")


=== Model Performance ===

Train Set:
  RMSE: 2.59
  MAE: 1.78
  R²: 0.994

Test Set:
  RMSE: 26.03
  MAE: 20.61
  R²: 0.393


## 5. Analyze Feature Importance


In [140]:
# Get feature importance
importance_df = pd.DataFrame(
    {"feature": all_features, "importance": model.feature_importances_}
).sort_values("importance", ascending=False)

# Categorize features
importance_df["type"] = importance_df["feature"].apply(
    lambda x: "LLM" if x in llm_features else "Statistical"
)

print("\n=== Feature Importance ===")
print(importance_df.to_string(index=False))

# Calculate aggregate importance by type
print("\n=== Importance by Feature Type ===")
importance_by_type = importance_df.groupby("type")["importance"].sum()
print(importance_by_type)
print(f"\nLLM features contribution: {importance_by_type.get('LLM', 0):.1%}")
print(
    f"Statistical features contribution: {importance_by_type.get('Statistical', 0):.1%}"
)


=== Feature Importance ===
                feature  importance        type
         avg_carries_3w    0.379914 Statistical
           press_rating    0.144211         LLM
        intuition_grade    0.088397         LLM
   avg_rushing_yards_3w    0.076415 Statistical
    prev_fantasy_points    0.060189 Statistical
opponent_defense_rating    0.059846         LLM
           oline_health    0.046561         LLM
           prev_carries    0.044315 Statistical
     prev_rushing_yards    0.041870 Statistical
        vegas_sentiment    0.030895         LLM
         injury_concern    0.027388         LLM

=== Importance by Feature Type ===
type
LLM            0.397297
Statistical    0.602703
Name: importance, dtype: float32

LLM features contribution: 39.7%
Statistical features contribution: 60.3%


In [141]:
import plotly.graph_objects as go

# Create horizontal bar chart
colors = ["#FF6B6B" if t == "LLM" else "#4ECDC4" for t in importance_df["type"]]

fig = go.Figure(
    data=[
        go.Bar(
            y=importance_df["feature"],
            x=importance_df["importance"],
            orientation="h",
            marker=dict(color=colors),
            text=importance_df["importance"].round(3),
            textposition="auto",
        )
    ]
)

fig.update_layout(
    title="XGBoost Feature Importance: LLM vs Statistical Features",
    xaxis_title="Importance",
    yaxis_title="Feature",
    height=500,
    showlegend=False,
)

fig.show()

## 6. Compare: Model With vs Without LLM Features


In [142]:
# Train baseline model without LLM features
model_baseline = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
)

X_train_baseline = X_train[stat_features]
X_test_baseline = X_test[stat_features]

model_baseline.fit(X_train_baseline, y_train)
y_pred_baseline = model_baseline.predict(X_test_baseline)

# Compare performance
print("\n=== Model Comparison ===")
print(f"\nBaseline (Statistical Features Only):")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_baseline)):.2f}")
print(f"  Test MAE: {mean_absolute_error(y_test, y_pred_baseline):.2f}")
print(f"  Test R²: {r2_score(y_test, y_pred_baseline):.3f}")

print(f"\nEnhanced (Statistical + LLM Features):")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}")
print(f"  Test MAE: {mean_absolute_error(y_test, y_pred_test):.2f}")
print(f"  Test R²: {r2_score(y_test, y_pred_test):.3f}")

rmse_improvement = np.sqrt(mean_squared_error(y_test, y_pred_baseline)) - np.sqrt(
    mean_squared_error(y_test, y_pred_test)
)
print(f"\nRMSE Improvement: {rmse_improvement:.2f} yards")


=== Model Comparison ===

Baseline (Statistical Features Only):
  Test RMSE: 33.82
  Test MAE: 24.98
  Test R²: -0.025

Enhanced (Statistical + LLM Features):
  Test RMSE: 26.03
  Test MAE: 20.61
  Test R²: 0.393

RMSE Improvement: 7.80 yards


In [143]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Calculate metrics for both models
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
rmse_enhanced = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
mae_enhanced = mean_absolute_error(y_test, y_pred_test)
r2_baseline = r2_score(y_test, y_pred_baseline)
r2_enhanced = r2_score(y_test, y_pred_test)

# Create subplots: 1 row, 2 columns
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Model Performance Metrics", "Prediction Accuracy Scatter"),
    specs=[[{"type": "bar"}, {"type": "scatter"}]],
)

# Subplot 1: Bar chart comparing metrics
# Add annotations to indicate direction
metrics = [
    "RMSE<br>(lower is better)",
    "MAE<br>(lower is better)",
    "R²<br>(higher is better)",
]
baseline_values = [rmse_baseline, mae_baseline, r2_baseline]
enhanced_values = [rmse_enhanced, mae_enhanced, r2_enhanced]

fig.add_trace(
    go.Bar(
        name="Baseline (Stats Only)",
        x=metrics,
        y=baseline_values,
        marker_color="#4ECDC4",
        text=[f"{v:.2f}" for v in baseline_values],
        textposition="outside",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        name="Enhanced (Stats + LLM)",
        x=metrics,
        y=enhanced_values,
        marker_color="#FF6B6B",
        text=[f"{v:.2f}" for v in enhanced_values],
        textposition="outside",
    ),
    row=1,
    col=1,
)

# Subplot 2: Scatter plot of actual vs predicted
fig.add_trace(
    go.Scatter(
        x=y_test,
        y=y_pred_baseline,
        mode="markers",
        name="Baseline",
        marker=dict(color="#4ECDC4", size=8, opacity=0.6),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=y_test,
        y=y_pred_test,
        mode="markers",
        name="Enhanced",
        marker=dict(color="#FF6B6B", size=8, opacity=0.6),
    ),
    row=1,
    col=2,
)

# Add perfect prediction line
max_yards = max(y_test.max(), y_pred_baseline.max(), y_pred_test.max())
fig.add_trace(
    go.Scatter(
        x=[0, max_yards],
        y=[0, max_yards],
        mode="lines",
        name="Perfect Prediction",
        line=dict(color="gray", dash="dash"),
    ),
    row=1,
    col=2,
)

# Update layout
fig.update_xaxes(title_text="Metric", row=1, col=1)
fig.update_yaxes(title_text="Value", row=1, col=1)
fig.update_xaxes(title_text="Actual Rushing Yards", row=1, col=2)
fig.update_yaxes(title_text="Predicted Rushing Yards", row=1, col=2)

fig.update_layout(
    title_text="Impact of LLM Feature Engineering on Prediction Accuracy",
    showlegend=True,
    height=500,
    width=1200,
)

fig.show()

# Print improvement summary
print("\n=== Improvement Summary ===")
print(
    f"RMSE Improvement: {rmse_baseline - rmse_enhanced:.2f} yards ({(rmse_baseline - rmse_enhanced)/rmse_baseline*100:.1f}%) ⬇️"
)
print(
    f"MAE Improvement: {mae_baseline - mae_enhanced:.2f} yards ({(mae_baseline - mae_enhanced)/mae_baseline*100:.1f}%) ⬇️"
)
print(
    f"R² Improvement: {r2_enhanced - r2_baseline:.3f} ({(r2_enhanced - r2_baseline)/abs(r2_baseline)*100:.1f}%) ⬆️"
)


=== Improvement Summary ===
RMSE Improvement: 7.80 yards (23.0%) ⬇️
MAE Improvement: 4.37 yards (17.5%) ⬇️
R² Improvement: 0.418 (1651.8%) ⬆️


## Week 5 Predictions


In [144]:
# Week 5 matchups as provided by the user
week_5_matchups = {
    "SF": "LA",  # San Francisco 49ers @ Los Angeles Rams (Thursday) - using 'LA' to match data
    "MIN": "CLE",  # Minnesota Vikings @ Cleveland Browns
    "NYG": "NO",  # New York Giants @ New Orleans Saints
    "HOU": "BAL",  # Houston Texans @ Baltimore Ravens
    "DEN": "PHI",  # Denver Broncos @ Philadelphia Eagles
    "DAL": "NYJ",  # Dallas Cowboys @ New York Jets
    "LV": "IND",  # Las Vegas Raiders @ Indianapolis Colts
    "MIA": "CAR",  # Miami Dolphins @ Carolina Panthers
    "TEN": "ARI",  # Tennessee Titans @ Arizona Cardinals
    "TB": "SEA",  # Tampa Bay Buccaneers @ Seattle Seahawks
    "DET": "CIN",  # Detroit Lions @ Cincinnati Bengals
    "WAS": "LAC",  # Washington Commanders @ Los Angeles Chargers
    "NE": "BUF",  # New England Patriots @ Buffalo Bills
    "KC": "JAX",  # Kansas City Chiefs @ Jacksonville Jaguars
}

print("Week 5 NFL Matchups loaded:")
for team, opponent in week_5_matchups.items():
    print(f"  {team} @ {opponent}")
print(f"\nTotal matchups: {len(week_5_matchups)}")

Week 5 NFL Matchups loaded:
  SF @ LA
  MIN @ CLE
  NYG @ NO
  HOU @ BAL
  DEN @ PHI
  DAL @ NYJ
  LV @ IND
  MIA @ CAR
  TEN @ ARI
  TB @ SEA
  DET @ CIN
  WAS @ LAC
  NE @ BUF
  KC @ JAX

Total matchups: 14


## Week 5 Predictions


In [145]:
# Week 5 matchups (ensuring consistent team abbreviations)
week_5_matchups = {
    "SF": "LA",  # San Francisco 49ers @ Los Angeles Rams (Thursday) - using 'LA' to match data
    "MIN": "CLE",  # Minnesota Vikings @ Cleveland Browns
    "NYG": "NO",  # New York Giants @ New Orleans Saints
    "HOU": "BAL",  # Houston Texans @ Baltimore Ravens
    "DEN": "PHI",  # Denver Broncos @ Philadelphia Eagles
    "DAL": "NYJ",  # Dallas Cowboys @ New York Jets
    "LV": "IND",  # Las Vegas Raiders @ Indianapolis Colts
    "MIA": "CAR",  # Miami Dolphins @ Carolina Panthers
    "TEN": "ARI",  # Tennessee Titans @ Arizona Cardinals
    "TB": "SEA",  # Tampa Bay Buccaneers @ Seattle Seahawks
    "DET": "CIN",  # Detroit Lions @ Cincinnati Bengals
    "WAS": "LAC",  # Washington Commanders @ Los Angeles Chargers
    "NE": "BUF",  # New England Patriots @ Buffalo Bills
    "KC": "JAX",  # Kansas City Chiefs @ Jacksonville Jaguars
}

print("Week 5 NFL Matchups loaded:")
for team, opponent in week_5_matchups.items():
    print(f"  {team} @ {opponent}")
print(f"\nTotal matchups: {len(week_5_matchups)}")

Week 5 NFL Matchups loaded:
  SF @ LA
  MIN @ CLE
  NYG @ NO
  HOU @ BAL
  DEN @ PHI
  DAL @ NYJ
  LV @ IND
  MIA @ CAR
  TEN @ ARI
  TB @ SEA
  DET @ CIN
  WAS @ LAC
  NE @ BUF
  KC @ JAX

Total matchups: 14


In [146]:
# Prepare week 5 prediction dataset using week 4 as base
# NOTE: We need to reload full data since rb_stats was filtered to weeks 2-4 only
print("Reloading full dataset to access week 4 data for all teams...")
weekly_stats_polars_full = nflread.load_player_stats([2025])
weekly_stats_full = pd.DataFrame(weekly_stats_polars_full.to_dict(as_series=False))
rb_stats_full = weekly_stats_full[weekly_stats_full["position"] == "RB"].copy()
rb_stats_full = rb_stats_full[stat_columns].copy()
rb_stats_full = rb_stats_full.dropna(subset=["rushing_yards"])

week_5_candidates = []

# Create inverse mapping for home teams (values in week_5_matchups become keys)
week_5_matchups_inverse = {v: k for k, v in week_5_matchups.items()}

# Combine both mappings so we have all teams
all_teams_matchups = {}
for away_team, home_team in week_5_matchups.items():
    all_teams_matchups[away_team] = home_team
    all_teams_matchups[home_team] = away_team

week_4_data = rb_stats_full[rb_stats_full["week"] == 4].copy()

for idx, row in week_4_data.iterrows():
    player_id = row["player_id"]
    player_name = row["player_name"]
    team = row["team"]

    # Check if team has week 5 matchup (either home or away)
    if team not in all_teams_matchups:
        continue

    opponent = all_teams_matchups[team]

    # Get player history from full dataset
    player_history = rb_stats_full[rb_stats_full["player_id"] == player_id].sort_values(
        "week"
    )

    if len(player_history) == 0:
        continue

    # Get week 4 for lagged features
    week_4_stats = player_history[player_history["week"] == 4]
    if len(week_4_stats) == 0:
        continue
    week_4_stats = week_4_stats.iloc[0]

    # Rolling avg from weeks 2-4
    weeks_2_to_4 = player_history[player_history["week"].isin([2, 3, 4])]

    pred_row = {
        "player_id": player_id,
        "player_name": player_name,
        "team": team,
        "opponent_team": opponent,
        "prev_rushing_yards": week_4_stats["rushing_yards"],
        "prev_carries": week_4_stats["carries"],
        "prev_fantasy_points": week_4_stats["fantasy_points_ppr"],
        "avg_rushing_yards_3w": (
            weeks_2_to_4["rushing_yards"].mean()
            if len(weeks_2_to_4) > 0
            else week_4_stats["rushing_yards"]
        ),
        "avg_carries_3w": (
            weeks_2_to_4["carries"].mean()
            if len(weeks_2_to_4) > 0
            else week_4_stats["carries"]
        ),
    }

    week_5_candidates.append(pred_row)

week_5_df = pd.DataFrame(week_5_candidates)
print(f"Week 5 prediction candidates: {len(week_5_df)} RBs")
print(f"Teams represented: {sorted(week_5_df['team'].unique())}")

# Check if we have LA predictions
if "LA" in week_5_df["team"].values:
    la_rbs = week_5_df[week_5_df["team"] == "LA"]["player_name"].tolist()
    print(f"\n✅ LA RBs included: {la_rbs}")
else:
    print(f"\n⚠️ No LA RBs found in prediction set")

Reloading full dataset to access week 4 data for all teams...
Week 5 prediction candidates: 80 RBs
Teams represented: ['ARI', 'BAL', 'BUF', 'CAR', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'SEA', 'SF', 'TB', 'TEN', 'WAS']

✅ LA RBs included: ['K.Williams', 'B.Corum']


In [147]:
# Generate LLM features for week 5 predictions
print(f"Generating LLM features for {len(week_5_df)} week 5 RBs...")
print("This may take a few minutes...\n")


async def get_week_5_llm_features(row):
    """Get all 6 LLM features for week 5 prediction"""
    player_name = row["player_name"]
    week = 5
    year = 2025
    opponent = row["opponent_team"]
    team = row["team"]

    # Get historical data for intuition grade (weeks 2-4) from full dataset
    player_history = rb_stats_full[
        (rb_stats_full["player_id"] == row["player_id"])
        & (rb_stats_full["week"].isin([2, 3, 4]))
    ][["week", "rushing_yards", "carries", "fantasy_points_ppr"]].tail(4)

    player_data_json = player_history.to_json(orient="records")

    # Run all 6 feature calls in parallel
    results = await asyncio.gather(
        get_llm_press_rating_async(player_name, week, year),
        get_llm_injury_concern_async(player_name, week, year),
        get_llm_intuition_grade_async(player_data_json),
        get_llm_opponent_defense_rating_async(opponent, week, year),
        get_llm_oline_health_async(team, week, year),
        get_llm_vegas_sentiment_async(player_name, week, year),
        return_exceptions=True,
    )

    # Handle exceptions
    default_values = [5, 1, 3, 5, 4, 5]
    processed = []
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            processed.append(default_values[i])
        else:
            processed.append(result)

    return processed


async def process_week_5_rows(df, max_concurrent=5):
    """Process all rows with concurrency limit"""
    semaphore = asyncio.Semaphore(max_concurrent)

    async def process_with_sem(idx, row):
        async with semaphore:
            player_name = row["player_name"]
            print(
                f"  Processing {player_name} - {row['team']} @ {row['opponent_team']}"
            )
            results = await get_week_5_llm_features(row)
            return idx, results

    tasks = [process_with_sem(idx, row) for idx, row in df.iterrows()]
    results = await asyncio.gather(*tasks)
    return results


# Run async processing
week_5_results = await process_week_5_rows(week_5_df, max_concurrent=5)

# Add LLM features to dataframe
feature_names = [
    "press_rating",
    "injury_concern",
    "intuition_grade",
    "opponent_defense_rating",
    "oline_health",
    "vegas_sentiment",
]

for idx, feature_values in week_5_results:
    for feature_name, value in zip(feature_names, feature_values):
        week_5_df.at[idx, feature_name] = value

print(f"\n✓ LLM features generated for {len(week_5_df)} RBs")

# Show LA RBs if they exist
if "LA" in week_5_df["team"].values:
    print(f"\n✅ LA RB predictions:")
    print(
        week_5_df[week_5_df["team"] == "LA"][
            ["player_name", "press_rating", "opponent_defense_rating"]
        ].to_string(index=False)
    )

Generating LLM features for 80 week 5 RBs...
This may take a few minutes...

  Processing R.Mostert - LV @ IND
  Processing D.Henry - BAL @ HOU
  Processing C.McCaffrey - SF @ LA
  Processing S.Perine - CIN @ DET
  Processing A.Kamara - NO @ NYG
  [CACHE HIT] Loaded press_rating from cache
  [CACHE HIT] Loaded injury_concern from cache
  [CACHE HIT] Loaded intuition_grade from cache
  [CACHE HIT] Loaded opponent_defense from cache
  [CACHE HIT] Loaded oline_health from cache
  [CACHE HIT] Loaded vegas_sentiment from cache
  [CACHE HIT] Loaded press_rating from cache
  [CACHE HIT] Loaded injury_concern from cache
  [CACHE HIT] Loaded intuition_grade from cache
  [CACHE HIT] Loaded oline_health from cache
  [CACHE HIT] Loaded vegas_sentiment from cache
  Processing K.Hunt - KC @ JAX
  [CACHE HIT] Loaded press_rating from cache
  [CACHE HIT] Loaded injury_concern from cache
  [CACHE HIT] Loaded intuition_grade from cache
  [CACHE HIT] Loaded opponent_defense from cache
  [CACHE HIT] Loade

In [148]:
# Make week 5 predictions using BOTH trained XGBoost models
week_5_features = week_5_df[all_features]

# Fill any missing values with defaults
week_5_features = week_5_features.fillna(
    {
        "press_rating": 5,
        "injury_concern": 1,
        "intuition_grade": 3,
        "opponent_defense_rating": 5,
        "oline_health": 4,
        "vegas_sentiment": 5,
    }
)

# Make predictions with BOTH models
# 1. Baseline model (statistical features only)
week_5_features_baseline = week_5_features[stat_features]
week_5_df["predicted_yards_baseline"] = model_baseline.predict(week_5_features_baseline)

# 2. Enhanced model (statistical + LLM features)
week_5_df["predicted_yards_enhanced"] = model.predict(week_5_features)

# Calculate the difference between models
week_5_df["prediction_difference"] = (
    week_5_df["predicted_yards_enhanced"] - week_5_df["predicted_yards_baseline"]
)

# Sort by enhanced model predictions and show top predictions
week_5_predictions = week_5_df[
    [
        "player_name",
        "team",
        "opponent_team",
        "prev_rushing_yards",
        "predicted_yards_baseline",
        "predicted_yards_enhanced",
        "prediction_difference",
        "press_rating",
        "injury_concern",
        "opponent_defense_rating",
    ]
].sort_values("predicted_yards_enhanced", ascending=False)

print("=== Week 5 RB Rushing Yards Predictions ===\n")
print("Top 15 Predicted Performances (Baseline vs Enhanced Models):\n")
print(week_5_predictions.head(15).to_string(index=False))

print(f"\n\nTotal predictions: {len(week_5_predictions)}")
print(
    f"\nAverage prediction difference (Enhanced - Baseline): {week_5_predictions['prediction_difference'].mean():.2f} yards"
)
print(
    f"Max positive impact from LLM features: {week_5_predictions['prediction_difference'].max():.2f} yards"
)
print(
    f"Max negative impact from LLM features: {week_5_predictions['prediction_difference'].min():.2f} yards"
)

=== Week 5 RB Rushing Yards Predictions ===

Top 15 Predicted Performances (Baseline vs Enhanced Models):

 player_name team opponent_team  prev_rushing_yards  predicted_yards_baseline  predicted_yards_enhanced  prediction_difference  press_rating  injury_concern  opponent_defense_rating
    A.Jeanty   LV           IND                 138                102.898193                109.158485               6.260292           9.0             1.0                      5.0
     J.Gibbs  DET           CIN                  91                 48.062943                106.262337              58.199394           9.0             1.0                      8.0
  C.Skattebo  NYG            NO                  79                125.881332                 86.932518             -38.948814           8.0             1.0                      5.0
   T.Pollard  TEN           ARI                  64                 67.828674                 80.637611              12.808937           5.0             1.0         

In [149]:
# Create Plotly visualization comparing baseline vs enhanced predictions
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Prepare data by team
teams_in_matchups = list(week_5_matchups.keys()) + list(week_5_matchups.values())
teams_in_matchups = sorted(set(teams_in_matchups))

# Get predictions grouped by team
team_predictions = {}
for team in teams_in_matchups:
    team_rbs = week_5_df[week_5_df["team"] == team].copy()
    if len(team_rbs) > 0:
        team_rbs = team_rbs.sort_values("predicted_yards_enhanced", ascending=False)
        team_predictions[team] = team_rbs

# Calculate grid dimensions
n_teams = len(team_predictions)
n_cols = 4
n_rows = (n_teams + n_cols - 1) // n_cols

# Create subplots - one for each team
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"{team} RBs" for team in sorted(team_predictions.keys())],
    specs=[[{"type": "bar"} for _ in range(n_cols)] for _ in range(n_rows)],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

# Add traces for each team
for idx, (team, team_rbs) in enumerate(sorted(team_predictions.items())):
    row = (idx // n_cols) + 1
    col = (idx % n_cols) + 1

    # Get top 3 RBs for this team
    top_rbs = team_rbs.head(3)

    # Create grouped bar chart for baseline and enhanced
    player_names = top_rbs["player_name"].apply(lambda x: x.split(".")[-1])

    fig.add_trace(
        go.Bar(
            x=player_names,
            y=top_rbs["predicted_yards_baseline"],
            name="Baseline",
            marker_color="#4ECDC4",
            showlegend=(idx == 0),  # Only show legend on first subplot
            hovertemplate="<b>%{x}</b><br>Baseline: %{y:.1f} yards<extra></extra>",
        ),
        row=row,
        col=col,
    )

    fig.add_trace(
        go.Bar(
            x=player_names,
            y=top_rbs["predicted_yards_enhanced"],
            name="Enhanced",
            marker_color="#FF6B6B",
            showlegend=(idx == 0),  # Only show legend on first subplot
            hovertemplate="<b>%{x}</b><br>Enhanced: %{y:.1f} yards<extra></extra>",
        ),
        row=row,
        col=col,
    )

    # Update axes for this subplot
    fig.update_xaxes(tickangle=-45, row=row, col=col)
    max_y = max(
        top_rbs["predicted_yards_enhanced"].max(),
        top_rbs["predicted_yards_baseline"].max(),
    )
    fig.update_yaxes(range=[0, max(100, max_y * 1.2)], row=row, col=col)

# Update overall layout
fig.update_layout(
    title_text="Week 5 RB Rushing Yards Predictions by Team<br><sub>Baseline (Stats Only) vs Enhanced (Stats + LLM Features)</sub>",
    height=300 * n_rows,
    barmode="group",
    font=dict(size=10),
)

fig.show()

print(f"\nVisualization created for {len(team_predictions)} teams")
print(f"Blue bars = Baseline model (statistical features only)")
print(f"Red bars = Enhanced model (statistical + LLM features)")


Visualization created for 28 teams
Blue bars = Baseline model (statistical features only)
Red bars = Enhanced model (statistical + LLM features)


## Week 5 SF @ LA: Model Performance Evaluation

The SF vs LA game (Thursday Night Football) has concluded. Let's evaluate how both models performed.


In [150]:
# Load actual Week 5 results for SF @ LA game
print("Loading Week 5 actual results for SF @ LA game...")
week_5_actual_polars = nflread.load_player_stats([2025])
week_5_actual = pd.DataFrame(week_5_actual_polars.to_dict(as_series=False))

# Filter for RBs, week 5, SF and LA teams
sf_la_week_5_actual = week_5_actual[
    (week_5_actual["position"] == "RB")
    & (week_5_actual["week"] == 5)
    & (week_5_actual["team"].isin(["SF", "LA"]))
][
    [
        "player_name",
        "team",
        "opponent_team",
        "rushing_yards",
        "carries",
        "receptions",
        "fantasy_points_ppr",
    ]
].copy()

sf_la_week_5_actual = sf_la_week_5_actual.sort_values(
    ["team", "rushing_yards"], ascending=[True, False]
)

print("\nActual Week 5 Performance - SF @ LA:")
print(sf_la_week_5_actual.to_string(index=False))

# Get our predictions for both SF and LA RBs
sf_la_predictions = week_5_df[week_5_df["team"].isin(["SF", "LA"])][
    [
        "player_name",
        "team",
        "prev_rushing_yards",
        "predicted_yards_baseline",
        "predicted_yards_enhanced",
        "prediction_difference",
    ]
].copy()

print("\n\nOur Predictions - SF and LA RBs:")
print(sf_la_predictions.sort_values("team").to_string(index=False))

Loading Week 5 actual results for SF @ LA game...

Actual Week 5 Performance - SF @ LA:
player_name team opponent_team  rushing_yards  carries  receptions  fantasy_points_ppr
 K.Williams   LA            SF             65       14           8                31.1
    B.Corum   LA            SF             13        1           0                 1.3
C.McCaffrey   SF            LA             57       22           8                27.9
 B.Robinson   SF            LA             12        5           0                 1.2
 I.Guerendo   SF            LA              0        0           0                 0.0


Our Predictions - SF and LA RBs:
player_name team  prev_rushing_yards  predicted_yards_baseline  predicted_yards_enhanced  prediction_difference
 K.Williams   LA                  77                 49.736244                 32.432617             -17.303627
    B.Corum   LA                  21                 24.311411                 33.194248               8.882837
C.McCaffrey   SF   

In [151]:
# Merge predictions with actuals for both SF and LA RBs
sf_la_evaluation = sf_la_predictions.merge(
    sf_la_week_5_actual[["player_name", "team", "rushing_yards", "carries"]],
    on=["player_name", "team"],
    how="inner",
    suffixes=("_pred", "_actual"),
)

# Calculate errors for both models
sf_la_evaluation["baseline_error"] = (
    sf_la_evaluation["predicted_yards_baseline"] - sf_la_evaluation["rushing_yards"]
)
sf_la_evaluation["enhanced_error"] = (
    sf_la_evaluation["predicted_yards_enhanced"] - sf_la_evaluation["rushing_yards"]
)
sf_la_evaluation["baseline_abs_error"] = sf_la_evaluation["baseline_error"].abs()
sf_la_evaluation["enhanced_abs_error"] = sf_la_evaluation["enhanced_error"].abs()

print("=== Model Performance Evaluation: SF @ LA ===\n")
print("Predictions vs Actuals (Both Teams):")
print(
    sf_la_evaluation[
        [
            "team",
            "player_name",
            "rushing_yards",
            "predicted_yards_baseline",
            "predicted_yards_enhanced",
            "baseline_error",
            "enhanced_error",
        ]
    ]
    .sort_values("team")
    .to_string(index=False)
)

# Calculate aggregate metrics
print("\n\n=== Aggregate Metrics (All RBs) ===")
print(f"\nBaseline Model (Stats Only):")
print(
    f"  Mean Absolute Error: {sf_la_evaluation['baseline_abs_error'].mean():.2f} yards"
)
print(f"  Total Error: {sf_la_evaluation['baseline_error'].sum():.2f} yards")
print(f"  RMSE: {np.sqrt((sf_la_evaluation['baseline_error']**2).mean()):.2f} yards")

print(f"\nEnhanced Model (Stats + LLM):")
print(
    f"  Mean Absolute Error: {sf_la_evaluation['enhanced_abs_error'].mean():.2f} yards"
)
print(f"  Total Error: {sf_la_evaluation['enhanced_error'].sum():.2f} yards")
print(f"  RMSE: {np.sqrt((sf_la_evaluation['enhanced_error']**2).mean()):.2f} yards")

# Determine winner
mae_improvement = (
    sf_la_evaluation["baseline_abs_error"].mean()
    - sf_la_evaluation["enhanced_abs_error"].mean()
)
print(
    f"\n{'✅' if mae_improvement > 0 else '❌'} Enhanced model MAE improvement: {mae_improvement:.2f} yards"
)

if mae_improvement > 0:
    print(
        f"   The LLM-enhanced model performed {mae_improvement:.2f} yards better on average!"
    )
else:
    print(
        f"   The baseline model performed {abs(mae_improvement):.2f} yards better on average."
    )

# Break down by team
print("\n\n=== By Team Breakdown ===")
for team in ["LA", "SF"]:
    team_data = sf_la_evaluation[sf_la_evaluation["team"] == team]
    if len(team_data) > 0:
        print(f"\n{team}:")
        print(f"  Baseline MAE: {team_data['baseline_abs_error'].mean():.2f} yards")
        print(f"  Enhanced MAE: {team_data['enhanced_abs_error'].mean():.2f} yards")
        improvement = (
            team_data["baseline_abs_error"].mean()
            - team_data["enhanced_abs_error"].mean()
        )
        print(
            f"  {'✅' if improvement > 0 else '❌'} Improvement: {improvement:.2f} yards"
        )

=== Model Performance Evaluation: SF @ LA ===

Predictions vs Actuals (Both Teams):
team player_name  rushing_yards  predicted_yards_baseline  predicted_yards_enhanced  baseline_error  enhanced_error
  LA  K.Williams             65                 49.736244                 32.432617      -15.263756      -32.567383
  LA     B.Corum             13                 24.311411                 33.194248       11.311411       20.194248
  SF C.McCaffrey             57                 95.283463                 79.576080       38.283463       22.576080
  SF  B.Robinson             12                 32.192085                 34.434540       20.192085       22.434540


=== Aggregate Metrics (All RBs) ===

Baseline Model (Stats Only):
  Mean Absolute Error: 21.26 yards
  Total Error: 54.52 yards
  RMSE: 23.63 yards

Enhanced Model (Stats + LLM):
  Mean Absolute Error: 24.44 yards
  Total Error: 32.64 yards
  RMSE: 24.91 yards

❌ Enhanced model MAE improvement: -3.18 yards
   The baseline model perf

In [153]:
# Visualize prediction accuracy for SF @ LA game (both teams)
import plotly.graph_objects as go

# Prepare data for visualization - sort by team then player
sf_la_evaluation_sorted = sf_la_evaluation.sort_values(
    ["team", "rushing_yards"], ascending=[True, False]
)

players = [
    f"{row['team']}: {row['player_name'].split('.')[-1]}"
    for _, row in sf_la_evaluation_sorted.iterrows()
]
actuals = sf_la_evaluation_sorted["rushing_yards"].tolist()
baseline_preds = sf_la_evaluation_sorted["predicted_yards_baseline"].tolist()
enhanced_preds = sf_la_evaluation_sorted["predicted_yards_enhanced"].tolist()

fig = go.Figure()

# Add bars for each model and actual
fig.add_trace(
    go.Bar(
        name="Actual",
        x=players,
        y=actuals,
        marker_color="#2ECC71",
        text=[f"{v:.0f}" for v in actuals],
        textposition="outside",
    )
)

fig.add_trace(
    go.Bar(
        name="Baseline Model",
        x=players,
        y=baseline_preds,
        marker_color="#4ECDC4",
        text=[f"{v:.0f}" for v in baseline_preds],
        textposition="outside",
    )
)

fig.add_trace(
    go.Bar(
        name="Enhanced Model",
        x=players,
        y=enhanced_preds,
        marker_color="#FF6B6B",
        text=[f"{v:.0f}" for v in enhanced_preds],
        textposition="outside",
    )
)

fig.update_layout(
    title="Week 5 SF @ LA: Predicted vs Actual Rushing Yards (Both Teams)",
    xaxis_title="Player",
    yaxis_title="Rushing Yards",
    barmode="group",
    height=500,
    showlegend=True,
)

fig.show()

print("\n🟢 Green = Actual performance")
print("🔵 Blue = Baseline model prediction")
print("🔴 Red = Enhanced model prediction")


🟢 Green = Actual performance
🔵 Blue = Baseline model prediction
🔴 Red = Enhanced model prediction


### Analysis: SF @ LA Week 5 Complete Evaluation

**Overall Result: Baseline Model Wins by 3.18 yards MAE** ❌

The statistical-only baseline model outperformed the LLM-enhanced model across all 4 running backs, but the story is more nuanced when we examine each team.

---

## Complete Game Results

| Team | Player | Actual | Baseline Pred | Enhanced Pred | Baseline Error | Enhanced Error | Winner |
|------|--------|--------|---------------|---------------|----------------|----------------|--------|
| **LA** | **K.Williams** | **65** | 50 | 32 | -15 ✅ | -33 | **Baseline** |
| **LA** | B.Corum | 13 | 24 | 33 | +11 ✅ | +20 | **Baseline** |
| **SF** | **C.McCaffrey** | **57** | 95 | 80 | +38 | +23 ✅ | **Enhanced** |
| **SF** | B.Robinson | 12 | 32 | 34 | +20 ✅ | +22 | **Baseline** |

**Model Performance:**
- **Baseline**: 21.26 MAE, 23.63 RMSE
- **Enhanced**: 24.44 MAE, 24.91 RMSE
- **Winner**: Baseline by 3.18 yards MAE

---

## Key Insights

### 1. **Dramatically Different Performance by Team**

**LA Rams (Baseline crushed it):**
- Baseline MAE: 13.29 yards
- Enhanced MAE: 26.38 yards  
- **Baseline won by 13.09 yards** ✅

**San Francisco (Enhanced was better):**
- Baseline MAE: 29.24 yards
- Enhanced MAE: 22.51 yards
- **Enhanced won by 6.73 yards** ✅

### 2. **The K.Williams Problem**

K.Williams was the biggest driver of the overall result:
- **Actual**: 65 yards (lead back with 14 carries)
- **Baseline**: 50 yards (off by 15) - reasonable underestimate
- **Enhanced**: 32 yards (off by 33) - severe underestimate

The LLM features caused the model to dramatically underpredict Williams' strong performance. Possible reasons:
- Opponent defense rating for SF may have been too pessimistic
- Press rating or Vegas sentiment may have been too conservative
- Statistical momentum (66→66→94→77 yards weeks 1-4) better captured by baseline

### 3. **McCaffrey: Where Enhanced Shined**

- **Actual**: 57 yards  
- **Baseline**: 95 yards (off by 38) - massive overestimate
- **Enhanced**: 80 yards (off by 23) - still high but 15 yards closer ✅

LLM features successfully tempered the baseline's over-optimism, likely due to:
- Previous week performance (49 yards) suggesting regression
- Opponent defense rating helping calibrate expectations

### 4. **Backup RBs: Both Models Struggled**

**B.Robinson (SF)**: 12 actual vs 32-34 predicted - both overshot significantly
**B.Corum (LA)**: 13 actual vs 24-33 predicted - both overshot significantly

Neither model could predict the limited workload for backups (5 and 1 carries respectively).

---

## Why Baseline Won Overall

1. **K.Williams dominated the scoring**: His 33-yard error for enhanced vs 15-yard error for baseline was decisive
2. **3 out of 4 players**: Baseline was more accurate on K.Williams, B.Corum, and B.Robinson
3. **Statistical momentum**: Baseline better captured Williams' hot streak without LLM features adding noise

## Why This Is Still Valuable

Despite losing overall, the **enhanced model showed its value on high-volume lead backs**:
- Won on the most important prediction (McCaffrey with 22 carries)
- Provided 6.73 yards improvement for SF predictions
- Successfully tempered over-optimistic projections

**The Problem**: LLM features **hurt predictions on the LA side**, particularly severely underpredicting K.Williams.

---

## Takeaways

✅ **What Worked**: LLM features helped dampen over-optimistic projections (McCaffrey)  
❌ **What Failed**: LLM features caused severe underprediction on a hot RB (K.Williams)  
🤔 **Team-Specific**: Enhanced was +6.73 yards better for SF, -13.09 yards worse for LA  
📊 **Sample Size**: Still only 4 predictions from 1 game - need more data

**Hypothesis**: LLM features may be most valuable when tempering overly optimistic statistical projections, but can introduce excessive pessimism when all indicators suggest strong performance.